# Part A — Data Preprocessing
## ACE6313 Machine Learning — Food Waste (SDG 12)

**Author:** Student 1 (Data Preprocessing)

This notebook builds the preprocessing pipeline: cleaning, transformation, and reduction.
It outputs `clean_food_waste.csv` for Part B.


## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
drive.mount('/content/drive')

# Load the raw dataset
df = pd.read_csv('/content/drive/My Drive/Food Waste data and research - by country.csv')
print("Shape:", df.shape)
df.head()

In [ ]:
# Column names and types
df.info()

In [ ]:
# Summary statistics
df.describe()

## 2. Data Cleaning
Handle missing values, duplicates, and outliers.

In [ ]:
# Missing values — this dataset has none, but we verify and report it
print("Missing values per column:")
print(df.isnull().sum())

In [ ]:
# Duplicate rows
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
# Drop columns not useful for modelling
# Source = just URLs, M49 code = country ID number
df = df.drop(columns=["Source", "M49 code"])
df.head()

In [ ]:
# Outlier detection using the IQR method on the main target figure
col = "combined figures (kg/capita/year)"
Q1, Q3 = df[col].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
outlier_mask = (df[col] < lower) | (df[col] > upper)
print(f"Outliers found: {outlier_mask.sum()}")
print(df.loc[outlier_mask, ["Country", col]])

# Visualise with a boxplot
plt.figure(figsize=(8,2))
sns.boxplot(x=df[col])
plt.title("Combined Food Waste — Outlier Check")
plt.tight_layout()
plt.savefig("/content/drive/My Drive/outlier_boxplot.png", dpi=150)
plt.show()

# DECISION: document in the report whether you keep or remove these.
# By default we KEEP them (real countries, not data errors).

## 3. Data Transformation
Encoding, feature engineering, and scaling.

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Encode the target (ordered: Very Low < Low < Medium < High)
conf_map = {"Very Low Confidence": 0, "Low Confidence": 1,
            "Medium Confidence": 2, "High Confidence": 3}
df["Confidence_encoded"] = df["Confidence in estimate"].map(conf_map)
print(df["Confidence_encoded"].value_counts().sort_index())

In [ ]:
# Encode Region (nominal — label encoding for tree models)
le = LabelEncoder()
df["Region_encoded"] = le.fit_transform(df["Region"])
print(dict(zip(le.classes_, le.transform(le.classes_))))

In [ ]:
# Feature engineering: each sector's share of total waste
df["household_ratio"] = (df["Household estimate (kg/capita/year)"]
                         / df["combined figures (kg/capita/year)"])
df["retail_ratio"] = (df["Retail estimate (kg/capita/year)"]
                      / df["combined figures (kg/capita/year)"])
df["foodservice_ratio"] = (df["Food service estimate (kg/capita/year)"]
                           / df["combined figures (kg/capita/year)"])
df[["household_ratio","retail_ratio","foodservice_ratio"]].head()

In [ ]:
# Feature scaling (StandardScaler) on numeric features
numeric_cols = [
    "combined figures (kg/capita/year)",
    "Household estimate (kg/capita/year)",
    "Retail estimate (kg/capita/year)",
    "Food service estimate (kg/capita/year)",
    "household_ratio", "retail_ratio", "foodservice_ratio",
]
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
df[numeric_cols].head()

## 4. Data Reduction
Correlation analysis + PCA.

In [ ]:
# Correlation heatmap to find redundant features
plt.figure(figsize=(10,8))
sns.heatmap(df[numeric_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("/content/drive/My Drive/correlation_heatmap.png", dpi=150)
plt.show()

In [ ]:
from sklearn.decomposition import PCA

# PCA to reduce dimensionality
pca = PCA(n_components=4)
pca_features = pca.fit_transform(df[numeric_cols])

print("Explained variance per component:", pca.explained_variance_ratio_.round(3))
print("Cumulative:", np.cumsum(pca.explained_variance_ratio_).round(3))

# Scree plot
plt.figure(figsize=(7,4))
plt.plot(range(1, len(pca.explained_variance_ratio_)+1),
         np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Scree Plot")
plt.grid(True)
plt.tight_layout()
plt.savefig("/content/drive/My Drive/pca_scree.png", dpi=150)
plt.show()

## 5. Save Cleaned Dataset for Part B

In [ ]:
# Final feature set for modelling
features = numeric_cols + ["Region_encoded"]
clean = df[features + ["Confidence_encoded"]].dropna()

clean.to_csv("/content/drive/My Drive/clean_food_waste.csv", index=False)
print("Saved clean_food_waste.csv with shape:", clean.shape)
clean.head()

---
### Summary (write this up in the report)
- **Cleaning:** verified no missing values, no duplicates; dropped Source/M49; checked outliers via IQR.
- **Transformation:** encoded target & region, engineered 3 sector-ratio features, scaled with StandardScaler.
- **Reduction:** correlation heatmap + PCA (note how many components capture ~95% variance).
